In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import __init__ as chess
from __init__ import Board
from svg import board

In [5]:
import os

pieces_dir = 'piece'
piece_sets = os.listdir(pieces_dir)

- alpha - good
- anarcandy - good
- cburnett - good
- chessnut - good
- companion - good
- fresca - good
- governor - good
- monarcy - good
- mono - good
- mpchess - good
- pirouetti - good
- pixel - good
- reillycraig - good
- riohacha - good
- shapes - good
- staunty - good
- tatiana - good
- caliente - good
- california - good
- cardinal - good
- celtic - good
- dubronvy - good
- fantasy - good
- gioco - good
- leipzig - good 
- letter - good
- libra - good
- maestro - good
- merida - good
- monarchy - good
- cooke - good
- kosal - good
- spatial - good
- chess7 - good
- horsey - good
- icpieces - good
- kiwen-suwi - good

Below we make some tweaks to the actual SVG files to be able to add their raw content in the `<defs>` tag of the outer board's `<svg>`. To do this:
- `id` attributes must to be globally unique
- classes defined in `<style>` tags must (should) be globally unique
- the `viewBox` attribute needs to be specified
- `height` and `width` attributes are not required (they will always be sized by a `<use>` tag)

To solve the first two, the code below prepends each ID and class name with the piece "code", ex. `wP-` or `bN-`, within each file.

Then to solve the last two, it removes the `height` and `width` attributes and adds a `viewBox` attribute if it doesn't exist by setting it like `viewBox="0 0 [oldHeight] [oldWidth]"`.

In [78]:
def make_ids_unique(svg_content: str, piece_code: str) -> str:
    """
    Read all SVG files in the given piece set dir. These should have names like
    'wP.svg', 'wN.svg', etc. (or 'P.svg', 'N.svg', etc. for the `mono` piece set).
    Prepend all appropriate `id` and `href` attributes in the SVG with the piece code
    and a hyphen (ex. 'wP-') to ensure global uniqueness when rendering an SVG with
    the full board. Then write the output to `output_piece_set_dir`.
    """
    # Replace all `id` and `href` attributes with the piece code prepended
    # `id`s will look like 'id="someId"` and the corresponding `href`s will look like 'href="#someId"'
    import re

    # Replace strings matching ex. `id="someId"` with `id="[pieceCode]-someId"
    svg_content = re.sub(r'id="([^"]+)"', f'id="{piece_code}-\\1"', svg_content)

    # Replace strings matching ex. `href="#someId"` with `href="#[pieceCode]-someId"
    svg_content = re.sub(r'href="#([^"]+)"', f'href="#{piece_code}-\\1"', svg_content)

    # Replace strings matching ex. `"url(#someId)"` with `"url(#[pieceCode]-someId)"`
    svg_content = re.sub(r'url\(#([^"]+)\)', f'url(#{piece_code}-\\1)', svg_content)

    # Get a list of all strings matching `id="..."` and verify that they are all unique
    ids = re.findall(r'id="([^"]+)"', svg_content)
    assert len(ids) == len(set(ids)), f'Non-unique IDs found for piece set {piece_code}'

    return svg_content

def make_classes_unique(svg_content: str, piece_code: str) -> str:
    """
    """
    import re

    # Replace all class names in style blocks (`<style>...</style>`) with `[pieceCode]-[className]`.
    # Also make a list of the class names that are being replaced so we can change where they are used.
    style_blocks = re.findall(r'<style>(.*?)</style>', svg_content, re.DOTALL)

    if len(style_blocks) == 0:
        return svg_content

    assert len(style_blocks) == 1, f'Expected exactly one <style> block in the SVG file:\n{svg_content}'
    style_block = style_blocks[0]

    # Get a list of all class names in the style block
    class_names = re.findall(r'\.([a-zA-Z_][a-zA-Z0-9_-]*)', style_block)

    # Replace all class names in the style block with `[pieceCode]-[className]`
    for class_name in class_names:
        old_class_name_def = f'.{class_name}'
        new_class_name_def = f'.{piece_code}-{class_name}'
        style_block = style_block.replace(old_class_name_def, new_class_name_def)
        
    # Replace the original style block with the modified one
    svg_content = svg_content.replace(style_blocks[0], style_block)

    # Replace the old class names with the new ones (`oldClassName` -> `[pieceCode]-oldClassName`).
    # Do a negative lookbehind to ensure we don't match any strings where the previous character is
    # a dot (ex. `.oldClassName`) or a hypen (where it may have already been replaced by ex. `wP-oldClassName`).
    for class_name in class_names:
        old_class_name = f'(?<![-.]){class_name}'
        new_class_name = f'{piece_code}-{class_name}'
        svg_content = re.sub(old_class_name, new_class_name, svg_content)
    
    return svg_content

def width_height_to_viewbox(svg_content: str) -> str:
    """
    Remove width and height attributes from the SVG, and if a viewBox attribute doesn't already exist,
    create it by setting it to the value "0 0 [oldWidth] [oldHeight]".
    """

    # Get the outer SVG element
    import xml.etree.ElementTree as ET
    root = ET.fromstring(svg_content)

    # Get the width and height attributes
    width = root.get('width')
    height = root.get('height')

    # Get the viewBox attribute
    viewbox = root.get('viewBox')

    # If the viewBox attribute doesn't exist, create it
    if viewbox is None:
        assert width is not None and height is not None, 'Both width/height and viewBox attributes are missing from the SVG'
        viewbox = f'0 0 {width} {height}'
        root.set('viewBox', viewbox)
    
    # Remove the width and height attributes
    if width is not None:
        assert height is not None, 'Width attribute exists but height attribute is missing'
        root.attrib.pop('width')
        root.attrib.pop('height')

    # Serialize the XML back to a string
    return ET.tostring(root, encoding='unicode')


for piece_set in piece_sets:
    svg_files = [f for f in os.listdir(os.path.join('piece', piece_set)) if f.endswith('.svg')]
    
    for svg_file in svg_files:
        piece_code = os.path.splitext(svg_file)[0]
        with open(os.path.join('piece', piece_set, svg_file), 'r') as f:
            svg_content = f.read()
        
        svg_content = make_ids_unique(svg_content, piece_code)
        svg_content = make_classes_unique(svg_content, piece_code)
        svg_content = width_height_to_viewbox(svg_content)

        with open(os.path.join('new_piece', piece_set, svg_file), 'w') as f:
            f.write(svg_content)

In [6]:
i = 0
b = Board()
b.push_san('e4')
b.push_san('c6')
b.push_san('d4')
b.push_san('d5')
b.push_san('e5')
b.push_san('Bf5')
b.push_san('Nf3')
b.push_san('g6')
b.push_san('Bd3')
b.push_san('Bg7')
b.push_san('O-O');

In [11]:
piece_set = piece_sets[i]
i += 1

print(f'Piece set: {piece_set}')
board(b, piece_set=piece_set, size=400, lastmove=b.peek())

Piece set: caliente


In [10]:
from svg import piece, board
import cairosvg


PIECE_SET = 'caliente'
piece_svg = piece(chess.Piece(chess.PAWN, chess.WHITE), piece_set=PIECE_SET)
board_svg = board(b, piece_set=PIECE_SET)
print(f'test:\n', board_svg)
png_data = cairosvg.svg2png(bytestring=board_svg)


test:
 <svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xlink="http://www.w3.org/1999/xlink" viewBox="0 0 390 390"><desc><pre>r n . q k . n r
p p . . p p b p
. . p . . . p .
. . . p P b . .
. . . P . . . .
. . . B . N . .
P P P . . P P P
R N B Q . R K .</pre></desc><defs><svg viewBox="0 0 16.933331 16.933331" id="white-pawn"><defs><linearGradient id="wP-b"><stop offset="0" stop-color="#fff" /></linearGradient><linearGradient id="wP-c" gradientTransform="translate(-6e-8 2.1166665)"><stop offset="0" stop-color="#ccc" /></linearGradient><linearGradient id="wP-a"><stop offset="0" stop-opacity=".2" /></linearGradient><linearGradient xlink:href="#wP-a" id="wP-d" x1="4.2333326" x2="103.04872" y1="24.341663" y2="24.341663" gradientTransform="matrix(1 0 0 1.25 .79374989 -3.0427069)" gradientUnits="userSpaceOnUse" /><linearGradient xlink:href="#wP-b" id="wP-e" x1="4.7624993" x2="12.170832" y1="9.5249987" y2="9.5249987" gradientUnits="userSpaceOnUse" /><lin

ParseError: duplicate attribute: line 1, column 83 (<string>)